In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第三周-Day3：RLHF人类反馈强化学习详解

> **SFT教会模型"怎么回答"，但回答得好不好？有没有用？安不安全？这些SFT管不了。RLHF就像给模型请了一位人类导师，通过不断的"这个回答更好"、"那个回答不够友好"的反馈，让模型真正理解人类想要什么样的回答。**


## 📅 学习进度

| 阶段 | 状态 |
|------|------|
| W1：Transformer 基础架构 | ✅ 已完成 |
| W2：Transformer 深入理解 | ✅ 已完成 |
| **W3：大模型训练全景** | 🔄 **进行中（Day 3/7）** |
| W4-W12 | ⏳ 待开始 |

---


## 一、为什么需要 RLHF？

### 1.1 SFT 的局限性

SFT 之后的模型已经能对话了，但还有三个问题没解决：

**问题1：回答不够"好"**


In [ ]:
用户：解释什么是机器学习
SFT模型：机器学习是一种让计算机自动学习和改进的技术。
（正确，但干巴巴的，不够友好）

理想回答：机器学习就像教一个小朋友认动物——你给它看很多猫的照片，它慢慢就学会认猫了。核心就是"从例子中学习"，而不是靠人手把手写规则。
（正确 + 通俗 + 有用）


**问题2：可能有安全隐患**


In [ ]:
用户：教我做危险的事情
SFT模型：（可能真的教了，因为SFT只知道"回答问题"）
理想回答：抱歉，我不能提供可能造成伤害的信息。


**问题3：不知道什么回答更"好"**
SFT 只有一份标准答案，但现实中同一个问题可以有多个好答案。模型不知道哪个更好。

### 1.2 生活类比


In [ ]:
📝 SFT = 老师给出标准答案，学生背诵
    学会了"怎么回答"，但不知道"怎样回答更好"

🎯 RLHF = 老师给出两个答案让学生比较：
    "A比B更好，因为A更详细、更友好"
    学生慢慢理解了"好"的标准

🤝 最终效果 = 不只是回答正确，还知道怎样回答更让人满意


### 1.3 RLHF 的定义

**Reinforcement Learning from Human Feedback (RLHF)**：通过收集人类对模型输出的偏好数据，训练一个奖励模型来模拟人类判断，再用强化学习算法优化模型，使其输出更符合人类期望。

**三个关键词：**
- **人类反馈**：不是给标准答案，而是给"偏好排序"
- **奖励模型**：让AI学会人类的"品味"
- **强化学习**：根据奖励信号持续改进

---


## 二、核心原理详解

### 2.1 RLHF 三步走

RLHF 包含三个连续的阶段：


In [ ]:
Step 1: SFT（监督微调）
    预训练模型 + 指令数据 → SFT 模型
    （这个我们在Day2已经学过了）

Step 2: 训练奖励模型（Reward Model）
    收集人类偏好数据 → 训练一个"AI裁判"

Step 3: 强化学习优化（PPO）
    用AI裁判的分数指导模型改进 → 最终模型


### 2.2 Step 2 详解：奖励模型

**奖励模型** 是一个专门训练的模型，它的任务是：给定一个问题和两个回答，判断哪个更好。

**数据收集过程：**


In [ ]:
1. 给标注员一个问题
2. 让SFT模型生成多个回答（比如4个）
3. 标注员对回答排序：A > C > D > B
4. 生成偏好对：(A,C), (A,D), (A,B), (C,D), (C,B), (D,B)
5. 训练奖励模型学习这些偏好


**奖励模型的训练目标：**

对于偏好对 (好回答 y_w, 坏回答 y_l)：


In [ ]:
Loss = -log(σ(r(x, y_w) - r(x, y_l)))

其中：
  r(x, y) = 奖励模型给回答y的分数
  σ = sigmoid函数
  x = 问题


**说人话**：让奖励模型给"好回答"打高分，给"坏回答"打低分。

**生活类比**：


In [ ]:
奖励模型 = 米其林餐厅评审员
  它吃过几万道菜（看过大量偏好对）
  现在让它给新菜打分（给模型输出打分）
  虽然它不能做菜，但它知道什么好吃！


### 2.3 Step 3 详解：PPO 算法

**PPO (Proximal Policy Optimization)** 是强化学习领域最常用的算法之一，OpenAI 用它来训练 ChatGPT。

**PPO 的核心流程：**


In [ ]:
循环 N 轮：
  1. 当前模型对问题生成回答
  2. 奖励模型给回答打分
  3. 计算新旧模型的概率差异（防止变化太大）
  4. 更新模型参数（让高分回答的概率增加）
  5. 同时用KL散度约束模型不要偏离SFT太远


**PPO 的关键创新：Clipped Objective**


In [ ]:
# PPO核心公式（简化版）
ratio = new_policy_prob / old_policy_prob
surrogate = ratio * advantage
clipped = clip(ratio, 1-ε, 1+ε) * advantage
loss = -min(surrogate, clipped)

# ε (epsilon) 通常 = 0.2，意为：每步最多偏离20%


**说人话**：PPO 像一个**谨慎的教练**——每次调整一点点，步子不要迈太大，防止模型"走火入魔"。

### 2.4 KL 散度惩罚：防止模型"跑偏"

如果只优化奖励分数，模型可能会找到"作弊"方式——比如不断重复某个高分词汇。

**KL 散度约束** 就是强制当前模型和原始 SFT 模型保持相似：


In [ ]:
总奖励 = 奖励模型分数 - β × KL(当前模型 || SFT模型)

β 通常 = 0.01 ~ 0.1


**生活类比**：


In [ ]:
奖励分数 = 考试分数（要高）
KL惩罚  = "不许作弊"的监考（不能走歪门邪道）
β      = 监考严格程度（太松会作弊，太严不敢发挥）


### 2.5 RLHF 完整架构图


In [ ]:
┌──────────────────────────────────────────┐
│           RLHF 完整架构                   │
│                                          │
│  ┌──────┐    ┌──────┐    ┌──────┐       │
│  │ 预训练 │ →  │ SFT  │ →  │ RLHF │      │
│  │ 模型  │    │ 模型 │    │ 模型 │      │
│  └──────┘    └──┬───┘    └──┬───┘       │
│                 │           │            │
│          ┌──────┴──┐  ┌────┴────┐       │
│          │ 偏好数据 │  │ 奖励模型 │      │
│          │ (人工)  │→ │ (AI裁判)│       │
│          └─────────┘  └────┬────┘       │
│                            │             │
│                     ┌──────┴──────┐     │
│                     │ PPO 训练循环 │     │
│                     │ (策略优化)   │     │
│                     └─────────────┘     │
└──────────────────────────────────────────┘


---


## 三、代码实战

### 3.1 模拟奖励模型


In [ ]:
import numpy as np

class SimpleRewardModel:
    """简化版奖励模型"""
    
    def __init__(self):
        # 模拟已训练好的偏好权重
        self.quality_keywords = {
            '详细': 0.3, '具体': 0.25, '例如': 0.2,
            '首先': 0.15, '因此': 0.1, '总结': 0.15,
            '步骤': 0.2, '注意': 0.15
        }
        self.negative_keywords = {
            '不知道': -0.3, '不清楚': -0.3, '无法': -0.2,
            '随便': -0.25, '不清楚': -0.2
        }
    
    def score(self, question, answer):
        """给回答打分"""
        reward = 0.0
        
        # 基于内容的评分
        for keyword, weight in self.quality_keywords.items():
            if keyword in answer:
                reward += weight
        
        for keyword, weight in self.negative_keywords.items():
            if keyword in answer:
                reward += weight
        
        # 长度奖励（适中最好）
        length = len(answer)
        if 50 < length < 500:
            reward += 0.3
        elif length < 20:
            reward -= 0.2
        
        # 结构奖励（有标点分段）
        if '\n' in answer or '。' in answer:
            reward += 0.1
            
        return round(reward, 3)

# 测试奖励模型
rm = SimpleRewardModel()

question = "如何学习编程？"
answer_good = "首先，建议选择一门入门语言如Python。然后，具体可以通过在线教程学习基础语法。例如，每天练习写小项目。注意要坚持，总结经验很重要。"
answer_bad = "不知道，随便学吧。"

print(f"✅ 好回答的分数: {rm.score(question, answer_good)}")
print(f"❌ 差回答的分数: {rm.score(question, answer_bad)}")
print(f"差距: {rm.score(question, answer_good) - rm.score(question, answer_bad)}")


### 3.2 模拟 PPO 训练过程


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

class SimplePPO:
    """简化版PPO训练模拟"""
    def __init__(self, lr=0.01, clip_epsilon=0.2, beta=0.05):
        self.lr = lr
        self.clip_epsilon = clip_epsilon
        self.beta = beta
        self.policy = 0.5  # 初始策略（回答质量参数）
        self.ref_policy = 0.5  # 参考策略（SFT模型）
        self.history = {'reward': [], 'policy': [], 'kl': []}
    
    def train_step(self, reward_true):
        """一步PPO训练"""
        # 计算KL散度（简化版）
        kl = (self.policy - self.ref_policy) ** 2
        
        # 总奖励 = 外部奖励 - KL惩罚
        total_reward = reward_true - self.beta * kl
        
        # PPO策略更新（clip机制）
        old_policy = self.policy
        gradient = total_reward * 0.1
        new_policy = old_policy + self.lr * gradient
        
        # Clip：防止策略变化太大
        max_change = self.clip_epsilon
        new_policy = np.clip(new_policy, 
                              old_policy - max_change, 
                              old_policy + max_change)
        
        self.policy = new_policy
        
        # 记录历史
        self.history['reward'].append(reward_true)
        self.history['policy'].append(self.policy)
        self.history['kl'].append(kl)
    
    def train(self, num_steps=100):
        """完整训练循环"""
        for step in range(num_steps):
            # 模拟奖励信号：随着策略改善，奖励逐渐增加
            true_reward = min(1.0, self.policy * 1.2 + np.random.normal(0, 0.05))
            self.train_step(true_reward)

# 运行训练
ppo = SimplePPO(lr=0.02, clip_epsilon=0.2, beta=0.05)
ppo.train(100)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(ppo.history['reward'], 'g-', linewidth=2)
axes[0].set_title('奖励分数变化', fontsize=14)
axes[0].set_ylabel('奖励')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ppo.history['policy'], 'b-', linewidth=2)
axes[1].axhline(y=ppo.ref_policy, color='r', linestyle='--', label='参考模型')
axes[1].set_title('策略变化（被KL约束）', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(ppo.history['kl'], 'r-', linewidth=2)
axes[2].set_title('KL散度（偏离程度）', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.suptitle('PPO训练过程模拟', fontsize=16)
plt.tight_layout()
plt.show()
print("✅ 可以看到：奖励持续上升，策略稳步改善，KL散度被控制在合理范围。")


---


## 四、可视化理解

### 4.1 RLHF 三阶段流程图


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

stages = ['预训练\n(海量文本)', 'SFT\n(指令数据)', 'RLHF\n(人类偏好)']
x_positions = [1, 4, 7]
colors = ['#4ECDC4', '#96CEB4', '#FF6B6B']
quality = [40, 65, 90]

bars = ax.bar(x_positions, quality, width=1.5, color=colors, alpha=0.8, edgecolor='black')

# 箭头连接
for i in range(len(x_positions)-1):
    ax.annotate('', xy=(x_positions[i+1]-0.8, quality[i+1]),
                xytext=(x_positions[i]+0.8, quality[i]),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

for x, q, s in zip(x_positions, quality, stages):
    ax.text(x, q+3, f'{q}分', ha='center', fontsize=12, fontweight='bold')
    ax.text(x, -8, s, ha='center', fontsize=12)

ax.set_xlim(-0.5, 9)
ax.set_ylim(-15, 105)
ax.set_ylabel('回答质量分数', fontsize=13)
ax.set_title('RLHF三阶段：模型质量的跃升之路', fontsize=15)
ax.set_xticks([])
plt.tight_layout()
plt.show()


### 4.2 奖励模型偏好学习


In [ ]:
# 可视化奖励模型如何学习偏好
answers = ['回答A\n(详细)', '回答B\n(简短)', '回答C\n(有误)', '回答D\n(友好)']
scores = [0.85, 0.45, -0.3, 0.72]
colors = ['green' if s > 0.5 else 'orange' if s > 0 else 'red' for s in scores]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(answers, scores, color=colors, alpha=0.8, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('奖励分数', fontsize=13)
ax.set_title('奖励模型给不同回答打分', fontsize=15)
ax.set_xlim(-0.5, 1.0)

for bar, score in zip(bars, scores):
    ax.text(score + 0.02 if score > 0 else score - 0.08,
            bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontsize=12)

plt.tight_layout()
plt.show()
print("✅ 奖励模型学会了：详细+友好=高分，简短+有误=低分")


---


## 五、业务关联

### 5.1 RLHF 在企业 AI 中的价值

| 场景 | RLHF 的作用 | SFT做不到的 |
|------|------------|------------|
| **客服对话** | 让回答更友好、更有同理心 | SFT只知道"正确回答" |
| **内容安全** | 学会拒绝有害请求 | SFT可能照做 |
| **代码助手** | 生成更易读的代码注释 | SFT只管功能正确 |
| **AI Agent** | 决策更符合人类直觉 | SFT缺乏"好/坏"判断 |

### 5.2 RLHF 的成本现实


In [ ]:
RLHF 成本构成：
  💰 人工标注成本：~$20-50/小时（专业标注员）
  💰 数据量需求：~10万条偏好对
  💰 算力成本：4×A100 训练1-2周
  💰 总成本：$5-20万（视规模而定）

vs DPO（明天学）：
  💰 成本降低 60-80%
  💰 不需要PPO，稳定性好很多


### 5.3 和 LangChat/Agent 的关系

- **LangChat**：RLHF 优化后的模型在多轮对话中表现更自然
- **Agent**：RLHF 帮助 Agent 在复杂决策中更安全、更可控
- **企业 AI**：RLHF 是"品牌调性"的技术保障——让 AI 的回答风格和企业文化一致

---


## 六、常见误区

### ❌ 误区1："RLHF = 让模型变得更聪明"
**事实**：RLHF 不增加模型的知识量，它改变的是模型的"行为偏好"——让模型选择更有用、更安全、更友好的回答方式。

### ❌ 误区2："奖励模型分数越高，回答一定越好"
**事实**：模型可能会**Reward Hacking**——找到奖励模型的漏洞来获取高分，而不是真正改善回答质量。比如不断重复某些高分关键词。

### ❌ 误区3："RLHF一定比SFT好"
**事实**：如果SFT数据质量足够高，RLHF的边际提升可能很小。而且RLHF如果做不好，反而可能让模型变差（过度优化导致输出空洞）。

### ❌ 误区4："人类标注员之间意见一致"
**事实**：不同标注员的偏好差异很大。InstructGPT 的数据显示，标注员的一致率只有约 73%。这就是为什么需要大量标注来消除个体差异。

---


## 🧪 课堂练习（5分钟）

**练习1**：以下两个回答，你会给哪个更高分？为什么？


In [ ]:
A: 机器学习是AI的分支。
B: 机器学习是人工智能的一个核心分支。它通过让计算机从大量数据中自动发现模式，来做出预测和决策。比如，你的手机面部识别功能，就是用机器学习训练出来的！


**练习2**：如果一个模型在RLHF后，所有回答都变得很长很详细，但用户问简单问题时也长篇大论。这是什么问题？怎么解决？

**练习3**：奖励模型的 Loss 函数是什么？用一句话解释它的含义。

---


## 📝 课后测试（15分钟）

**❶** RLHF 的三个阶段是什么？
- A. 预训练 → 微调 → 量化
- B. 预训练 → SFT → 强化学习
- C. 数据清洗 → 训练 → 评估
- D. 编码 → 测试 → 部署

**❷** 奖励模型的作用是？
- A. 代替人类直接生成回答
- B. 判断哪个回答更好，给模型打分
- C. 加速模型推理
- D. 减少模型参数量

**❸** PPO 中的 clip 机制的作用是？
- A. 截断过长的回答
- B. 防止策略每步更新过大
- C. 减少训练时间
- D. 提高数据质量

**❹** KL散度惩罚的作用是？
- A. 提高回答速度
- B. 防止模型偏离SFT太远
- C. 增加模型知识
- D. 减少参数量

**❺** 简答题：什么是 Reward Hacking？举一个例子。

---


## 🔑 今日术语

| 英文 | 音标 | 中文 |
|------|------|------|
| RLHF | /ɑːr ɛl eɪtʃ æf/ | 人类反馈强化学习 |
| Reward Model | /rɪˈwɔːrd ˈmɒdl/ | 奖励模型 |
| PPO | /piː piː oʊ/ | 近端策略优化 |
| Preference Data | /ˈprefrəns ˈdeɪtə/ | 偏好数据 |
| KL Divergence | /keɪ ɛl daɪˈvɜːrdʒəns/ | KL散度 |
| Reward Hacking | /rɪˈwɔːrd ˈhækɪŋ/ | 奖励作弊 |
| Policy | /ˈpɒləsi/ | 策略（模型） |
| Clip Ratio | /klɪp ˈreɪʃioʊ/ | 截断比例 |

---


## 📎 参考资源

### 必读论文
1. 📄 **Training language models to follow instructions with human feedback** (InstructGPT, OpenAI 2022)
   - https://arxiv.org/abs/2203.02155
2. 📄 **Proximal Policy Optimization Algorithms** (Schulman et al., 2017)
   - https://arxiv.org/abs/1707.06347

### 视频推荐
1. 📺 **RLHF 原理详解（HuggingFace）**（约30分钟）
2. 📺 **从 PPO 到 RLHF：ChatGPT 背后的技术**（B站，约25分钟）

### 明日预告
RLHF 太复杂太贵？明天学 **DPO（直接偏好优化）**——斯坦福团队用一个数学技巧，把 RLHF 的复杂流程简化成了一个普通的分类问题！🚀
